# Writing your own `StepSolver` and `SystemSolver`

See [0_README.ipynb](0_README.ipynb) for the protocol overview and the
full shipped-solver landscape on both axes. This notebook writes one
custom solver on each axis — not toy reorderings, but each one
inverting a real, documented design decision already made by a shipped
solver, so the comparison teaches an actual trade-off rather than an
arbitrary difference:

- **Axis 1:** `SpeciationAfterFeedStepSolver` reverses
  `SequentialAdvanceSolver`'s documented ordering (see
  [`docs/dev/implementation/shipped/ORDERING.md`](../../../docs/dev/implementation/shipped/ORDERING.md)) —
  the shipped default pins speciation to the *pre-feed* state within
  one step, so a large feed's pH-shifting effect only becomes visible
  on the *next* step. This solver feeds first, so the same-step pH
  already reflects the feed.
- **Axis 2:** `AsymmetricThreeStageSystemSolver` follows the same
  extension pattern `StrangSplittingSystemSolver` established
  (`sim._apply_links` / `cv._advance_unchecked` /
  `sim._invoke_controllers`), just composed into an asymmetric 3-stage
  split instead of one symmetric half-step.

In [1]:
from PyOMES.core import (
    AdvanceResult,
    AdvectiveLink,
    ControlVolume,
    LiquidPhase,
    SequentialAdvanceSolver,
    Simulation,
)
from PyOMES.core.system_solver import ExplicitEulerSystemSolver
from PyOMES.reactions import EquilibriumReaction, ReactionSystem, StoichiometryEntry
from PyOMES.chemistry.common_species import H_plus, OH_minus, H2O
from typing import Any, Dict, Optional

print("Imports OK")


Imports OK


## 1  Axis 1 — a custom `StepSolver`

`SequentialAdvanceSolver` (the shipped default — `solver=None` is pure
sugar for it) runs speciation *before* applying that step's feed, so
the feed's effect on pH is deliberately invisible until the *next*
step (the "operator-splitting contract" in `ORDERING.md`). Reversing
that order is a one-class change:

```
1. External source terms + boundary fluxes (feed)   <- moved first
2. Speciation solve                                  <- now post-feed
3. Reactions (post-feed, post-speciation state)
4. Internal transfer
```

Neither ordering is "more correct" in the abstract — which one you
want depends on what you're reproducing: PyOMES's own convention, or an
external reference model that feeds first.

In [2]:
class SpeciationAfterFeedStepSolver:
    """Feed the CV *before* solving speciation, reversing
    ``SequentialAdvanceSolver``'s documented pre-feed-speciation
    contract.

    This is the minimum viable custom ``StepSolver`` -- one method,
    ``solve_step(cv, dt_h, t_h, external_source_terms)``, returning an
    ``AdvanceResult``. Everything else (the shipped rungs, this one,
    and any solver a user writes) satisfies the exact same protocol.
    """

    def solve_step(
        self,
        cv: Any,
        dt_h: float,
        t_h: float = 0.0,
        external_source_terms: Optional[Dict[str, Dict[str, float]]] = None,
    ) -> AdvanceResult:
        # 1. Feed first.
        if external_source_terms:
            for phase_key, rates in external_source_terms.items():
                if phase_key in cv.phases:
                    cv.phases[phase_key].apply_flux(rates, dt_h)
        for boundary in cv.boundaries:
            flux = boundary.compute_flux(cv, dt_h)
            if boundary.phase_key in cv.phases:
                cv.phases[boundary.phase_key].apply_flux(flux, dt_h)

        # 2. Speciation on the POST-feed state (the reversed step).
        if cv.reaction_system is not None and hasattr(cv.reaction_system, "engine"):
            engine = cv.reaction_system.engine
            if engine is not None:
                liq = cv.phases.get("liquid")
                if liq is not None:
                    result = engine.solve(phases=cv.phases, T_K=float(liq.T_K))
                    result.apply_to_phases(cv.phases)

        # 3. Reactions, against the now-post-feed speciation.
        rxn_sources = None
        if cv.reaction_system is not None:
            rhs = cv.compute_reaction_rates(t_h)
            rxn_sources = {}
            for pk, sp_rates in rhs.items():
                rxn_sources[pk] = {}
                phase = cv.phases[pk]
                for sp, rate in sp_rates.items():
                    phase.n_mol[sp] = max(0.0, phase.n_mol.get(sp, 0.0) + rate * dt_h)
                    rxn_sources[pk][sp] = rate

        # 4. Internal transfer last.
        transfer_diag = cv.step_internal_transfer(dt_h)

        return AdvanceResult(transfer=transfer_diag, reaction_sources=rxn_sources)


def make_acid_base_cv() -> ControlVolume:
    """A liquid CV with a water-dissociation equilibrium -- enough to
    show a pH shift from a strong-base feed within a single step."""
    liq = LiquidPhase(n_mol={"Na+": 0.0}, V_L=1.0, T_K=298.15)
    rxn = EquilibriumReaction(
        stoichiometry=[
            StoichiometryEntry(species=H2O, phase="liquid", coefficient=-1.0),
            StoichiometryEntry(species=H_plus, phase="liquid", coefficient=+1.0),
            StoichiometryEntry(species=OH_minus, phase="liquid", coefficient=+1.0),
        ],
        log_K=-14.0,
        balance_elements=("H", "O"),
        label="water",
    )
    system = ReactionSystem([rxn], label="water_equilibrium")
    return ControlVolume(phases={"liquid": liq}, reaction_system=system, label="cv")

print("SpeciationAfterFeedStepSolver defined")


SpeciationAfterFeedStepSolver defined


### Run it

A strong-base feed (`Na+`/`OH-`) large enough to noticeably shift pH
within `dt_h`, applied under both the shipped default and the custom
ordering, starting from the same initial state.

In [3]:
feed = {"liquid": {"Na+": 0.02, "OH-": 0.02}}  # mol/h, applied over dt_h

cv_default = make_acid_base_cv()
cv_custom = make_acid_base_cv()

# Seed derived species (H+/OH-) on a fresh CV via one speciation solve,
# so pH is readable before either custom ordering runs.
for cv in (cv_default, cv_custom):
    engine = cv.reaction_system.engine
    result = engine.solve(phases=cv.phases, T_K=float(cv.phases["liquid"].T_K))
    result.apply_to_phases(cv.phases)
pH_before = cv_default.phases["liquid"].pH

cv_default.advance(
    dt_h=1.0, t_h=0.0, external_source_terms=feed,
    solver=SequentialAdvanceSolver(),
)
cv_custom.advance(
    dt_h=1.0, t_h=0.0, external_source_terms=feed,
    solver=SpeciationAfterFeedStepSolver(),
)

print(f"pH before the step:                          {pH_before:.3f}")
print(f"pH after (SequentialAdvanceSolver, shipped):  {cv_default.phases['liquid'].pH:.3f}")
print(f"pH after (SpeciationAfterFeedStepSolver):     {cv_custom.phases['liquid'].pH:.3f}")


pH before the step:                          7.000
pH after (SequentialAdvanceSolver, shipped):  7.000
pH after (SpeciationAfterFeedStepSolver):     12.301


Both are valid Lie-Trotter splittings of the same physics — they
differ only in which state the same-step speciation solve sees.
`SequentialAdvanceSolver`'s choice is documented in `ORDERING.md`;
reproducing a reference model that feeds first is a one-class change,
not a fork of the framework.

## 2  Axis 2 — a custom `SystemSolver`

`StrangSplittingSystemSolver` upgrades `ExplicitEulerSystemSolver` to
2nd-order accuracy with one extra `_apply_links` call:
`links(dt/2) -> cv.advance(dt) -> links(dt/2)`. A bespoke `SystemSolver`
is no more than composing the same three primitives
(`sim._apply_links` / `cv._advance_unchecked` /
`sim._invoke_controllers`) in whatever order your model — or the
reference model you're reproducing — needs. Here: an asymmetric
3-stage split instead of one symmetric half-step.

In [4]:
class AsymmetricThreeStageSystemSolver:
    """Bespoke 3-stage inter-CV interleaving: an asymmetric link split
    around two half-duration CV advances, instead of
    ``StrangSplittingSystemSolver``'s single symmetric half-step.

    Sequence per macro step::

        profiles(t) -> links(dt*w0) -> cv.advance(dt/2)
                    -> links(dt*w1) -> cv.advance(dt/2)
                    -> links(dt*w2) -> controllers

    with asymmetric weights ``w0 + w1 + w2 == 1`` (default
    ``(0.2, 0.5, 0.3)``).
    """

    def __init__(self, weights=(0.2, 0.5, 0.3)):
        if len(weights) != 3 or abs(sum(weights) - 1.0) > 1e-9:
            raise ValueError("weights must be a 3-tuple summing to 1.0")
        self.weights = weights

    def advance_system(self, sim: "Simulation", dt_h: float, t_h: float) -> Any:
        w0, w1, w2 = self.weights
        profile_actions = sim._invoke_profiles(t_h=t_h)

        link_records = sim._apply_links(dt_h * w0)

        results: Dict[str, "AdvanceResult"] = {}
        for cv_key, cv in sim.cvs.items():
            solver = sim._solver_for(cv_key)
            results[cv_key] = cv._advance_unchecked(dt_h / 2.0, t_h, solver=solver)

        link_records += sim._apply_links(dt_h * w1)

        for cv_key, cv in sim.cvs.items():
            solver = sim._solver_for(cv_key)
            # Overwrites the first-half AdvanceResult -- callers that need
            # both would extend AdvanceResult or accumulate a list instead.
            results[cv_key] = cv._advance_unchecked(dt_h / 2.0, t_h + dt_h / 2.0, solver=solver)

        link_records += sim._apply_links(dt_h * w2)

        controller_actions = [
            a for a in sim._invoke_controllers(t_h=t_h, dt_h=dt_h, results=results)
            if a is not None
        ]
        for action in controller_actions:
            sim._apply_controller_action(action, dt_h=dt_h)

        return results, link_records, controller_actions, profile_actions


def build_two_cv_sim(system_solver=None) -> Simulation:
    liq_r = LiquidPhase(n_mol={"S": 10.0}, V_L=1.0, T_K=310.0)
    liq_c = LiquidPhase(n_mol={"S": 0.0}, V_L=1.0, T_K=310.0)
    return Simulation(
        cvs={
            "reactor": ControlVolume(phases={"liquid": liq_r}, label="reactor"),
            "recycle": ControlVolume(phases={"liquid": liq_c}, label="recycle"),
        },
        links=[
            AdvectiveLink(
                _source_cv_key="reactor", _source_phase_key="liquid",
                _sink_cv_key="recycle", _sink_phase_key="liquid",
                Q_L_per_h=2.0, _label="fwd",
            ),
            AdvectiveLink(
                _source_cv_key="recycle", _source_phase_key="liquid",
                _sink_cv_key="reactor", _sink_phase_key="liquid",
                Q_L_per_h=2.0, _label="bwd",
            ),
        ],
        system_solver=system_solver,
        label="custom_system_solver_demo",
    )

print("AsymmetricThreeStageSystemSolver defined")


AsymmetricThreeStageSystemSolver defined


### Run it

Same two-CV recirculating model (`reactor` <-> `recycle`) under
`ExplicitEulerSystemSolver` and the custom asymmetric split.

In [5]:
dt_h, n_steps = 0.1, 10

sim_euler = build_two_cv_sim(ExplicitEulerSystemSolver())
sim_custom = build_two_cv_sim(AsymmetricThreeStageSystemSolver())

result_euler = sim_euler.run(tau_h=dt_h * n_steps, n_steps=n_steps)
result_custom = sim_custom.run(tau_h=dt_h * n_steps, n_steps=n_steps)

S_euler = result_euler.liquid_mol["reactor"]["S"][-1]
S_custom = result_custom.liquid_mol["reactor"]["S"][-1]

print(f"ExplicitEulerSystemSolver          reactor S = {S_euler:.5f} mol")
print(f"AsymmetricThreeStageSystemSolver    reactor S = {S_custom:.5f} mol")
print(f"Relative difference: {abs(S_euler - S_custom) / S_euler:.2e}")


ExplicitEulerSystemSolver          reactor S = 5.60680 mol
AsymmetricThreeStageSystemSolver    reactor S = 5.27433 mol
Relative difference: 5.93e-02


Both conserve mass and converge to the same steady state; they differ
in splitting error at finite `dt_h` — exactly the same trade-off
`StrangSplittingSystemSolver` demonstrates against Euler, just with a
bespoke asymmetric weighting instead of a symmetric one.

## Summary

| Axis | Protocol | Shipped default | Custom solver here | What it changes |
|---|---|---|---|---|
| 1 — per-CV physics | `StepSolver.solve_step(cv, dt_h, t_h, external_source_terms)` | `SequentialAdvanceSolver` (speciation before feed) | `SpeciationAfterFeedStepSolver` | Same-step pH visibility of a feed |
| 2 — whole-system orchestration | `SystemSolver.advance_system(sim, dt_h, t_h)` | `ExplicitEulerSystemSolver` / `StrangSplittingSystemSolver` (symmetric split) | `AsymmetricThreeStageSystemSolver` | Inter-CV transport splitting error at finite `dt_h` |

Both protocols are one method each — a bespoke solver is exactly as
much code as the physics/orchestration it needs, no framework
subclassing required. See [0_README.ipynb](0_README.ipynb) for the
full landscape of shipped solvers on both axes.